# **Enrichment - Synthetic Data Generation**

A complete resource to generate synthetic data: fill **one column**, **two or more columns** or build an **entire dataset**. Every method comes with metrics to measure the quality of the generated data.

**Dataset:** `credit_data.csv` - 2,000 clients described by `income`, `age`, `loan` and the classifier `result` (original name `c#default`: `1` = the client defaulted).

| Section | Content |
|---|---|
| 00 | Import libs and the dataset |
| 01 | Enrich one column |
| 02 | Enrich two or more columns |
| 03 | Generate a complete synthetic dataset |
| 04 | Quality metrics of the synthetic data |
| 05 | Compare the methods and choose one |

### **00. Import Libs and the DataSet**

In [ ]:
#Import Libs
import pandas as pd
import numpy as np
import random
import string

In [ ]:
#Import Specific Libs
from scipy import stats
from scipy.spatial.distance import jensenshannon
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.model_selection import train_test_split, cross_val_predict, StratifiedKFold
from sklearn.neighbors import NearestNeighbors
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score
from IPython.display import display

In [ ]:
#Import the Graphic Libs
import matplotlib.pyplot as plt
import seaborn as sns

In [ ]:
#Ignore Error Message
import warnings
warnings.simplefilter(action='ignore', category=FutureWarning)

In [ ]:
#Import Dataframe from Github
df = pd.read_csv('https://raw.githubusercontent.com/jrafa1607/DataScience-ML-AI/main/-%20Datasets/credit_data.csv')

In [ ]:
#Define New Attributes Names
df.columns = ['id', 'income', 'age', 'loan', 'result']
df.head(3)

In [ ]:
#Set the Seed ( Same Seed = Same Synthetic Data )
SEED = 42
np.random.seed(SEED)
random.seed(SEED)

In [ ]:
#Prepare the Real Data ( Reference to Fit the Generators and to Compare the Results )
# Attention! The real data has 3 negative ages and 3 null ages. They are removed here only to build a clean reference ( see the Data Preprocessing notebook )
real = df.copy()
real.loc[real['age'] < 0, 'age'] = np.nan
real = real.dropna().reset_index(drop=True)
print(real.shape)
real.describe().round(2)

### **01. Enrich One Column**

Adding a column is assigning a list or an array with the **same length as the dataframe**: `df['new_column'] = values`. The examples below fill the same column in many different ways, and every example ends with the metrics that show if the generated values are what we designed.

In [ ]:
#Create the Dataframe to Receive the New Column
df_example = real.copy()
df_example['Data_Column'] = pd.NA
print(df_example.head())

In [ ]:
#Example 01 - Enrich with Random Integer Values (Between 1 and 101)
random_integers = np.random.randint(1, 101, size=len(df_example))
df_example['Data_Column'] = random_integers
print(df_example['Data_Column'].head(10))

#Metrics ( Range and Uniformity )
counts, _ = np.histogram(random_integers, bins=10, range=(1, 101))
print('\nMin / Max:', random_integers.min(), '/', random_integers.max())
print('Chi-square test of uniformity ( p-value > 0.05 = no evidence against a uniform distribution ):', round(stats.chisquare(counts).pvalue, 4))

In [ ]:
#Example 02 - Enrich with Random Float Values (Between 0 and 1)
random_floats = np.random.uniform(0, 1, size=len(df_example))
df_example['Data_Column'] = random_floats
print(df_example['Data_Column'].head(10))

#Metrics ( Kolmogorov-Smirnov Test vs Uniform Distribution )
ks = stats.kstest(random_floats, 'uniform')
print('\nKS statistic:', round(ks.statistic, 4), '| p-value:', round(ks.pvalue, 4), '( p-value > 0.05 = matches the uniform distribution )')

In [ ]:
#Example 03 - Enrich with Random String Values (Text)
def generate_random_string(length=10):
    letters = string.ascii_lowercase
    return ''.join(random.choice(letters) for i in range(length))

random_strings = [generate_random_string() for _ in range(len(df_example))]
df_example['Data_Column'] = random_strings
print(df_example['Data_Column'].head(10))

#Metrics ( Length, Alphabet and Uniqueness )
print('\nAll the strings have 10 letters:', (df_example['Data_Column'].str.len() == 10).all())
print('Only lowercase letters:', df_example['Data_Column'].str.fullmatch('[a-z]+').all())
print('Unique values (%):', round(df_example['Data_Column'].nunique() / len(df_example) * 100, 2))

In [ ]:
#Example 04 - Enrich with Random Boolean Values (True/False)
random_booleans = np.random.choice([True, False], size=len(df_example))
df_example['Data_Column'] = random_booleans
print(df_example['Data_Column'].head(10))

#Metrics ( Share of True and Binomial Test vs 50% )
n_true = int(df_example['Data_Column'].sum())
print('\nTrue (%):', round(n_true / len(df_example) * 100, 2), '| Binomial test p-value:', round(stats.binomtest(n_true, len(df_example), 0.5).pvalue, 4))

In [ ]:
#Example 05 - Enrich with Specific Values ( List Predefined )
values_list = ['Zero', 'One', 'Two', 'Three', 'Four', 'Five', 'Six', 'Seven', 'Eight', 'Nine']
df_example['Data_Column'] = np.resize(values_list, len(df_example))    # Repeat the list until the column is full
print(df_example['Data_Column'].head(12))

#Metrics ( Balanced Use of the Values )
print('\nMin / Max rows per value:', df_example['Data_Column'].value_counts().min(), '/', df_example['Data_Column'].value_counts().max())

In [ ]:
#Example 06 - Enrich with Specific Random Values ( List Predefined )
values_list = ['Zero', 'One', 'Two']
random_selections = np.random.choice(values_list, size=len(df_example))
df_example['Data_Column'] = random_selections
print(df_example['Data_Column'].head(10))

#Metrics ( Chi-Square Test vs Equal Probabilities )
counts = df_example['Data_Column'].value_counts()
print('\n', counts.to_dict())
print('p-value:', round(stats.chisquare(counts).pvalue, 4), '( p-value > 0.05 = every value has the same probability )')

In [ ]:
#Example 07 - Enrich with Random Values with Different Probabilities ( Weighted Choice )
status = ['Single', 'Married', 'Divorced', 'Widowed']
probabilities = [0.35, 0.50, 0.10, 0.05]
df_example['Data_Column'] = np.random.choice(status, size=len(df_example), p=probabilities)

#Metrics ( Observed vs Expected Proportions )
observed = df_example['Data_Column'].value_counts().reindex(status)
expected = np.array(probabilities) * len(df_example)
print(pd.DataFrame({'observed_%': (observed / len(df_example) * 100).round(2), 'expected_%': np.array(probabilities) * 100}))
print('\nChi-square goodness of fit ( p-value > 0.05 = matches the probabilities ):', round(stats.chisquare(observed, expected).pvalue, 4))

In [ ]:
#Example 08 - Enrich with Sequential Values ( Codes and IDs )
df_example['Data_Column'] = 'CL-' + pd.Series(np.arange(1, len(df_example) + 1)).astype(str).str.zfill(5)
print(df_example['Data_Column'].head())

#Metrics ( Unique Values and Format )
print('\nAll unique:', df_example['Data_Column'].is_unique, '| Format CL-00000:', df_example['Data_Column'].str.fullmatch(r'CL-\d{5}').all())

In [ ]:
#Example 09 - Enrich with a Constant Value
df_example['Data_Column'] = 'Brazil'

#Metrics ( One Single Value )
print(df_example['Data_Column'].value_counts().to_dict(), '| Unique values:', df_example['Data_Column'].nunique())

In [ ]:
#Example 10 - Enrich with Random Dates ( Between Two Dates )
start_date, end_date = pd.Timestamp('2020-01-01'), pd.Timestamp('2024-12-31')
random_days = np.random.randint(0, (end_date - start_date).days + 1, size=len(df_example))
df_example['Data_Column'] = start_date + pd.to_timedelta(random_days, unit='D')
print(df_example['Data_Column'].head())

#Metrics ( Period and Distribution per Year )
print('\nMin / Max:', df_example['Data_Column'].min().date(), '/', df_example['Data_Column'].max().date())
print((df_example['Data_Column'].dt.year.value_counts(normalize=True).sort_index() * 100).round(1).to_dict())

In [ ]:
#Example 11 - Enrich with Normal Distribution ( Mean 40 and Standard Deviation 10 )
mean, std = 40, 10
values = np.random.normal(loc=mean, scale=std, size=len(df_example))
df_example['Data_Column'] = np.clip(values, 18, 80)    # Clip to keep only valid values
print(df_example['Data_Column'].describe().round(2))

#Metrics ( KS Test vs Normal Distribution, Skewness and Kurtosis )
ks = stats.kstest(values, 'norm', args=(mean, std))
print('\nKS p-value:', round(ks.pvalue, 4), '| Skewness ( 0 = symmetric ):', round(stats.skew(values), 3), '| Excess kurtosis ( 0 = normal ):', round(stats.kurtosis(values), 3))

In [ ]:
#Example 12 - Enrich with Log-Normal Distribution ( Right Skewed, as Salaries and Loans )
median_value, sigma = 3000, 0.5
values = np.random.lognormal(mean=np.log(median_value), sigma=sigma, size=len(df_example))
df_example['Data_Column'] = values.round(2)
print(df_example['Data_Column'].describe().round(2))

#Metrics ( KS Test vs Log-Normal, Median and Skewness )
ks = stats.kstest(values, 'lognorm', args=(sigma, 0, median_value))
print('\nKS p-value:', round(ks.pvalue, 4), '| Median expected / observed:', median_value, '/', round(np.median(values), 1), '| Skewness ( > 0 = right skewed ):', round(stats.skew(values), 2))

In [ ]:
#Example 13 - Enrich with Poisson Distribution ( Counts, as Number of Calls or Purchases )
lam = 2
df_example['Data_Column'] = np.random.poisson(lam=lam, size=len(df_example))
print(df_example['Data_Column'].value_counts().sort_index().head(8).to_dict())

#Metrics ( Mean = Variance = Lambda )
print('\nMean:', round(df_example['Data_Column'].mean(), 3), '| Variance:', round(df_example['Data_Column'].var(), 3), '| Expected for both:', lam)

In [ ]:
#Example 14 - Enrich with Beta Distribution ( Rates and Percentages Between 0 and 1 )
a, b = 2, 5
values = np.random.beta(a, b, size=len(df_example))
df_example['Data_Column'] = values.round(4)
print(df_example['Data_Column'].describe().round(3))

#Metrics ( KS Test vs Beta and Mean )
ks = stats.kstest(values, 'beta', args=(a, b))
print('\nKS p-value:', round(ks.pvalue, 4), '| Mean expected / observed:', round(a / (a + b), 3), '/', round(values.mean(), 3))

In [ ]:
#Example 15 - Enrich with Random 0/1 Flags ( Bernoulli with Probability of 14% )
probability = 0.14
df_example['Data_Column'] = np.random.binomial(1, probability, size=len(df_example))

#Metrics ( Observed Rate and Binomial Test )
n_ones = int(df_example['Data_Column'].sum())
print('Rate expected / observed:', probability, '/', round(n_ones / len(df_example), 4), '| Binomial test p-value:', round(stats.binomtest(n_ones, len(df_example), probability).pvalue, 4))

In [ ]:
#Example 16 - Enrich Based on an Existing Column ( Bootstrap: Sample the Real Values with Replacement )
df_example['Data_Column'] = np.random.choice(real['age'].values, size=len(df_example), replace=True)

#Metrics ( Two-Sample KS Test vs the Real Column )
ks = stats.ks_2samp(real['age'], df_example['Data_Column'])
print('KS statistic:', round(ks.statistic, 4), '| p-value:', round(ks.pvalue, 4), '( p-value > 0.05 = same distribution as the real column )')

In [ ]:
#Example 17 - Enrich with a Derived Column ( Formula from Other Columns )
df_example['Data_Column'] = df_example['loan'] / df_example['income']
print(df_example['Data_Column'].describe().round(4))

#Metrics ( No Nulls and No Infinite Values )
print('\nNulls:', df_example['Data_Column'].isnull().sum(), '| Infinite:', np.isinf(df_example['Data_Column']).sum())

In [ ]:
#Example 18 - Enrich with a Derived Column with Noise ( Relationship Controlled by Us )
slope, noise_std = 0.30, 1500
noise = np.random.normal(0, noise_std, size=len(df_example))
df_example['Data_Column'] = (df_example['income'] * slope + noise).clip(lower=0).round(2)

#Metrics ( Achieved R2 vs Designed R2 )
r = df_example['Data_Column'].corr(df_example['income'])
signal_var = slope ** 2 * df_example['income'].var()
print('Correlation with income:', round(r, 3))
print('R2 designed / observed:', round(signal_var / (signal_var + noise_std ** 2), 3), '/', round(r ** 2, 3))

In [ ]:
#Example 19 - Enrich with Conditional Values ( Rules with np.select )
conditions = [(df_example['age'] < 30) & (df_example['loan'] > 5000),
              (df_example['age'] < 30) | (df_example['loan'] > 5000)]
choices = ['High', 'Medium']
df_example['Data_Column'] = np.select(conditions, choices, default='Low')
print(df_example['Data_Column'].value_counts().to_dict())

#Metrics ( Business Validity: the Default Rate Should Grow with the Risk Level )
print((df_example.groupby('Data_Column')['result'].mean().reindex(['Low', 'Medium', 'High']) * 100).round(2).to_dict())

In [ ]:
#Example 20 - Enrich with Categories from Numeric Ranges ( Binning with pd.cut )
df_example['Data_Column'] = pd.cut(df_example['age'], bins=[0, 30, 40, 50, 100], labels=['<=30', '31-40', '41-50', '51+'])
print(df_example['Data_Column'].value_counts().sort_index().to_dict())

#Metrics ( Every Row Received a Category )
print('Nulls:', df_example['Data_Column'].isnull().sum())

In [ ]:
#Example 21 - Enrich with Text Following a Pattern ( E-mails )
first_names = ['ana', 'bruno', 'carla', 'diego', 'elisa', 'fabio', 'gabi', 'hugo']
last_names = ['silva', 'souza', 'lima', 'costa', 'rocha', 'alves']
first = pd.Series(np.random.choice(first_names, size=len(df_example)))
last = pd.Series(np.random.choice(last_names, size=len(df_example)))
number = pd.Series(np.random.randint(1, 1000, size=len(df_example))).astype(str)
df_example['Data_Column'] = first + '.' + last + number + '@example.com'
print(df_example['Data_Column'].head())

#Metrics ( Valid Pattern and Duplicates )
print('\nValid e-mail pattern (%):', round(df_example['Data_Column'].str.fullmatch(r'[a-z]+\.[a-z]+\d+@example\.com').mean() * 100, 2))
print('Duplicated values (%):', round(df_example['Data_Column'].duplicated().mean() * 100, 2))

In [ ]:
#Example 22 - Enrich with Null Values ( Create Missing Data to Practice )
# Completely at random ( MCAR ): every row has the same chance of being null
df_example['age_with_nulls'] = real['age']
df_example.loc[np.random.rand(len(df_example)) < 0.05, 'age_with_nulls'] = np.nan

# Depends on another column ( MAR ): low income clients have more nulls
low_income = df_example['income'] < df_example['income'].median()
df_example['loan_with_nulls'] = real['loan']
df_example.loc[np.random.rand(len(df_example)) < np.where(low_income, 0.10, 0.02), 'loan_with_nulls'] = np.nan

#Metrics ( Observed Null Rate )
print('MCAR - null rate expected / observed (%):', 5, '/', round(df_example['age_with_nulls'].isnull().mean() * 100, 2))
print('MAR  - null rate low income / high income (%):', round(df_example.loc[low_income, 'loan_with_nulls'].isnull().mean() * 100, 2), '/', round(df_example.loc[~low_income, 'loan_with_nulls'].isnull().mean() * 100, 2), '( expected 10 / 2 )')

In [ ]:
#Example 23 - Enrich with Outliers ( Extreme Values at 6 Standard Deviations )
df_example['loan_with_outliers'] = real['loan']
outlier_rows = np.random.choice(len(df_example), size=int(0.01 * len(df_example)), replace=False)
df_example.loc[outlier_rows, 'loan_with_outliers'] = real['loan'].mean() + 6 * real['loan'].std()

#Metrics ( How Many Injected Outliers Does the IQR Rule Detect? )
q1, q3 = df_example['loan_with_outliers'].quantile([0.25, 0.75])
flagged = (df_example['loan_with_outliers'] > q3 + 1.5 * (q3 - q1)).values
print('Injected:', len(outlier_rows), '| Detected by IQR:', int(flagged[outlier_rows].sum()), '| Recall (%):', round(flagged[outlier_rows].mean() * 100, 1))

In [ ]:
#Example 24 - Fill ONLY the Null Values with Synthetic Values ( Random Draw from the Observed Values )
column = df_example['age_with_nulls']
null_mask = column.isnull()

filled_random = column.copy()
filled_random.loc[null_mask] = np.random.choice(column.dropna().values, size=null_mask.sum())    # Hot-deck: keeps the distribution
filled_mean = column.fillna(column.mean())                                                         # Mean: keeps only the mean

#Metrics ( Error vs the True Values and Distribution Preserved )
true_values = real['age']
for name, filled in [('Random draw', filled_random), ('Mean', filled_mean)]:
    rmse = np.sqrt(((filled[null_mask] - true_values[null_mask]) ** 2).mean())
    ks = stats.ks_2samp(filled, true_values).statistic
    print(f'{name:<12} RMSE: {rmse:6.2f} | Std: {filled.std():5.2f} ( true: {true_values.std():5.2f} ) | KS vs true: {ks:.4f}')

### **02. Enrich Two or More Columns**

When more than one column is generated, the **relationship between the columns** matters as much as each column alone. The examples below create independent columns, correlated columns, dependent categories and columns that follow the real relationships.

In [ ]:
#Create the Dataframe to Receive the New Columns
df_multi = real.copy()

In [ ]:
#Example 25 - Add Several Independent Columns at Once ( assign )
df_multi = df_multi.assign(Col_A=np.random.randint(1, 101, size=len(df_multi)),
                           Col_B=np.random.choice(['Red', 'Green', 'Blue'], size=len(df_multi)),
                           Col_C=np.random.normal(0, 1, size=len(df_multi)))
print(df_multi[['Col_A', 'Col_B', 'Col_C']].head())

#Metrics ( Independent Columns Must Have Correlation Close to Zero )
r, p_value = stats.pearsonr(df_multi['Col_A'], df_multi['Col_C'])
print('\nCorrelation Col_A x Col_C:', round(r, 3), '| p-value:', round(p_value, 4), '( p-value > 0.05 = independent )')

In [ ]:
#Example 26 - Two Numeric Columns with a Target Correlation ( Multivariate Normal )
target_corr = 0.70
covariance = [[1, target_corr], [target_corr, 1]]
pair = np.random.multivariate_normal([0, 0], covariance, size=len(df_multi))
df_multi['Col_D'] = (50 + 10 * pair[:, 0]).round(2)
df_multi['Col_E'] = (1000 + 200 * pair[:, 1]).round(2)

#Metrics ( Achieved Correlation vs Target and 95% Confidence Interval )
r = df_multi['Col_D'].corr(df_multi['Col_E'])
z_score = np.arctanh(r)
low, high = np.tanh(z_score - 1.96 / np.sqrt(len(df_multi) - 3)), np.tanh(z_score + 1.96 / np.sqrt(len(df_multi) - 3))
print('Target / achieved correlation:', target_corr, '/', round(r, 3), '| Absolute error:', round(abs(r - target_corr), 3))
print('95% confidence interval:', (round(low, 3), round(high, 3)), '| Contains the target:', low <= target_corr <= high)

In [ ]:
#Example 27 - A Column that Depends on Another ( Non Linear Relationship + Noise )
df_multi['Col_F'] = (np.exp(0.03 * df_multi['Col_A']) + np.random.normal(0, 0.3, size=len(df_multi))).round(3)

#Metrics ( Spearman Captures Any Monotonic Relationship, Pearson Only the Linear One )
print('Pearson:', round(df_multi['Col_A'].corr(df_multi['Col_F']), 3), '| Spearman:', round(df_multi['Col_A'].corr(df_multi['Col_F'], method='spearman'), 3))

In [ ]:
#Example 28 - Categories that Depend on Each Other ( Hierarchy: City -> State )
city_state = {'Sao Paulo': 'SP', 'Campinas': 'SP', 'Rio de Janeiro': 'RJ', 'Niteroi': 'RJ',
              'Belo Horizonte': 'MG', 'Uberlandia': 'MG', 'Curitiba': 'PR', 'Londrina': 'PR'}
df_multi['City'] = np.random.choice(list(city_state), size=len(df_multi), p=[0.25, 0.10, 0.20, 0.05, 0.15, 0.05, 0.15, 0.05])
df_multi['State'] = df_multi['City'].map(city_state)
print(df_multi[['City', 'State']].head())

#Metrics ( Consistency: Every City Has Only One State )
print('\nStates per city ( max ):', df_multi.groupby('City')['State'].nunique().max(), '| Nulls:', df_multi['State'].isnull().sum())

In [ ]:
#Example 29 - Categories with Conditional Probabilities ( Education Depends on the Age Group )
df_multi['age_group'] = pd.cut(df_multi['age'], bins=[0, 30, 45, 100], labels=['Young', 'Adult', 'Senior'])
levels = ['High School', 'Bachelor', 'Postgraduate']
designed = pd.DataFrame({'Young': [0.55, 0.40, 0.05], 'Adult': [0.35, 0.45, 0.20], 'Senior': [0.50, 0.30, 0.20]}, index=levels).T

df_multi['Education'] = ''
for group, probabilities in designed.iterrows():
    mask = df_multi['age_group'] == group
    df_multi.loc[mask, 'Education'] = np.random.choice(levels, size=mask.sum(), p=probabilities.values)

#Metrics ( Observed vs Designed Probabilities, Chi-Square Test and Cramer's V )
table = pd.crosstab(df_multi['age_group'], df_multi['Education'])
observed = pd.crosstab(df_multi['age_group'], df_multi['Education'], normalize='index')[levels]
chi2, p_value, _, _ = stats.chi2_contingency(table)
cramers_v = np.sqrt(chi2 / (table.values.sum() * (min(table.shape) - 1)))
print('Max absolute difference vs designed:', round((observed - designed).abs().values.max(), 3))
print("Chi-square p-value:", round(p_value, 6), "( p-value < 0.05 = the columns are dependent ) | Cramer's V:", round(cramers_v, 3))

In [ ]:
#Example 30 - A Numeric Column that Depends on a Category ( Bonus by Job )
jobs = ['Clerk', 'Analyst', 'Manager', 'Director']
mean_by_job = {'Clerk': 1500, 'Analyst': 3000, 'Manager': 6000, 'Director': 12000}
std_by_job = {'Clerk': 300, 'Analyst': 600, 'Manager': 1200, 'Director': 2400}
df_multi['Job'] = np.random.choice(jobs, size=len(df_multi), p=[0.40, 0.30, 0.20, 0.10])
df_multi['Bonus'] = np.random.normal(df_multi['Job'].map(mean_by_job).values, df_multi['Job'].map(std_by_job).values).clip(min=0).round(2)

#Metrics ( Group Means vs Designed, ANOVA and Effect Size )
print(pd.DataFrame({'designed_mean': pd.Series(mean_by_job), 'observed_mean': df_multi.groupby('Job')['Bonus'].mean().round(1)}))
groups = [g['Bonus'].values for _, g in df_multi.groupby('Job')]
f_stat, p_value = stats.f_oneway(*groups)
eta_squared = sum(len(g) * (g.mean() - df_multi['Bonus'].mean()) ** 2 for g in groups) / ((df_multi['Bonus'] - df_multi['Bonus'].mean()) ** 2).sum()
print('\nANOVA p-value:', round(p_value, 6), '| Eta squared ( share of variance explained by the job ):', round(eta_squared, 3))

In [ ]:
#Example 31 - Two Columns from the Real Data ( Joint Sampling Keeps the Relationship )
rows = np.random.randint(0, len(real), size=len(df_multi))
pair = real.loc[rows, ['income', 'loan']].reset_index(drop=True)
df_multi['income_synth'] = (pair['income'] + np.random.normal(0, 0.05 * real['income'].std(), len(df_multi))).clip(lower=0).values
df_multi['loan_synth'] = (pair['loan'] + np.random.normal(0, 0.05 * real['loan'].std(), len(df_multi))).clip(lower=0).values

#Metrics ( Correlation and Distribution vs the Real Columns )
print('Correlation income x loan - real / synthetic:', round(real['income'].corr(real['loan']), 3), '/', round(df_multi['income_synth'].corr(df_multi['loan_synth']), 3))
print('KS statistic - income:', round(stats.ks_2samp(real['income'], df_multi['income_synth']).statistic, 4), '| loan:', round(stats.ks_2samp(real['loan'], df_multi['loan_synth']).statistic, 4))

In [ ]:
#Example 32 - A Column Predicted by a Model ( Prediction + Residual Noise )
features = ['income', 'age']
model = RandomForestRegressor(n_estimators=100, min_samples_leaf=20, oob_score=True, random_state=SEED, n_jobs=-1).fit(real[features], real['loan'])
residuals = real['loan'].values - model.oob_prediction_
df_multi['loan_model'] = (model.predict(df_multi[features]) + np.random.choice(residuals, size=len(df_multi))).clip(min=real['loan'].min())

#Metrics ( Out-of-Bag R2, Residual Spread and Distribution vs the Real Column )
print('Out-of-bag R2 ( how much the features explain the loan ):', round(model.oob_score_, 3))
print('Std real / synthetic:', round(real['loan'].std(), 1), '/', round(df_multi['loan_model'].std(), 1), '| KS statistic:', round(stats.ks_2samp(real['loan'], df_multi['loan_model']).statistic, 4))

In [ ]:
#Example 33 - Add Several Columns from a Dictionary of Generators
column_spec = {
    'Customer_Code': lambda n: ('CL-' + pd.Series(np.arange(1, n + 1)).astype(str).str.zfill(5)).values,
    'Segment': lambda n: np.random.choice(['Retail', 'Corporate', 'SME'], size=n, p=[0.60, 0.10, 0.30]),
    'Score': lambda n: np.clip(np.random.normal(650, 80, n), 300, 850).round(),
    'Active': lambda n: np.random.binomial(1, 0.80, n).astype(bool)}

def add_columns(data, spec):
    out = data.copy()
    for name, generator in spec.items():
        out[name] = generator(len(out))
    return out

df_spec = add_columns(real, column_spec)
print(df_spec.head(3))

#Metrics ( Shape, Types and Nulls of the New Columns )
print('\nRows kept:', len(df_spec) == len(real), '| New columns:', list(column_spec), '| Nulls:', int(df_spec[list(column_spec)].isnull().sum().sum()))
print(df_spec[list(column_spec)].dtypes.astype(str).to_dict())

### **03. Generate a Complete Synthetic Dataset**

Eight ways to create a **full dataset** (all the columns at once), from the simplest to the most complete. The generators learn only from the **training part** of the real data, and the **test part** is kept aside to evaluate the result without leakage.

| Method | Idea | Keeps the correlations? | Needs real data? |
|---|---|---|---|
| 01 Independent Bootstrap | Sample each column alone | No | Yes |
| 02 Gaussian by Class | Normal distribution per class ( mean and covariance ) | Yes ( linear ) | Yes |
| 03 Gaussian Copula | Real distributions + real correlations | Yes | Yes |
| 04 KDE by Class | Smooth density estimated per class | Yes | Yes |
| 05 Smoothed Bootstrap | Sample real rows + random noise | Yes | Yes |
| 06 SMOTE-like | Interpolate between neighbor rows | Yes | Yes |
| 07 Chained Models | Each column predicted from the previous ones | Yes | Yes |
| 08 From Schema | Distributions and rules written by hand | Only if we write it | **No** |

In [ ]:
#Define the Attributes and Split the Real Data ( Train to Fit the Generators / Test to Evaluate )
num_cols = ['income', 'age', 'loan']
target = 'result'
real_train, real_test = train_test_split(real[num_cols + [target]], test_size=0.30, stratify=real[target], random_state=SEED)
real_train, real_test = real_train.reset_index(drop=True), real_test.reset_index(drop=True)
N_SYNTH = len(real_train)
print(real_train.shape, real_test.shape)

In [ ]:
#Define the Rules of a Valid Row and the Function that Enforces Them
rules = {'income > 0': lambda d: d['income'] > 0,
         'age between 18 and 100': lambda d: d['age'].between(18, 100),
         'loan >= 0': lambda d: d['loan'] >= 0,
         'loan < income': lambda d: d['loan'] < d['income'],
         'result is 0 or 1': lambda d: d['result'].isin([0, 1])}

def validity_rate(data):
    # Share of rows that follow ALL the rules
    return np.logical_and.reduce([rule(data) for rule in rules.values()]).mean() * 100

def apply_constraints(data, reference):
    out = data.copy()
    for col in num_cols:
        out[col] = out[col].clip(reference[col].min(), reference[col].max())
    out[target] = out[target].round().clip(0, 1).astype(int)
    out.insert(0, 'id', np.arange(1, len(out) + 1))
    return out[['id'] + num_cols + [target]]

synthetic, validity = {}, {}

def finalize(raw, name):
    validity[name] = validity_rate(raw)                  # Rules checked BEFORE the correction
    synthetic[name] = apply_constraints(raw, real_train)
    return synthetic[name]

In [ ]:
#Method 01 - Independent Bootstrap ( Baseline: Each Column is Sampled Alone )
def synth_independent(data, n, seed=SEED):
    rng = np.random.default_rng(seed)
    return pd.DataFrame({col: rng.choice(data[col].values, size=n, replace=True) for col in data.columns})

finalize(synth_independent(real_train, N_SYNTH), '01 Independent Bootstrap').head()

In [ ]:
#Method 02 - Gaussian by Class ( Mean and Covariance of Each Class )
def synth_gaussian_by_class(data, n, seed=SEED):
    rng = np.random.default_rng(seed)
    prior = data[target].value_counts(normalize=True).sort_index()
    counts = rng.multinomial(n, prior.values)
    parts = []
    for cls, k in zip(prior.index, counts):
        group = data.loc[data[target] == cls, num_cols]
        part = pd.DataFrame(rng.multivariate_normal(group.mean().values, group.cov().values, size=k), columns=num_cols)
        part[target] = cls
        parts.append(part)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

finalize(synth_gaussian_by_class(real_train, N_SYNTH), '02 Gaussian by Class').head()

In [ ]:
#Method 03 - Gaussian Copula ( Keeps the Real Distribution of Each Column and the Correlation Between Them )
def synth_gaussian_copula(data, n, seed=SEED):
    rng = np.random.default_rng(seed)
    cols = list(data.columns)
    # 1) Real values -> ranks -> normal scores
    normal_scores = stats.norm.ppf(data[cols].rank(method='average').values / (len(data) + 1))
    correlation = np.corrcoef(normal_scores, rowvar=False)
    # 2) New normal scores with the same correlation -> uniform values
    uniform = stats.norm.cdf(rng.multivariate_normal(np.zeros(len(cols)), correlation, size=n))
    # 3) Uniform values -> real values, using the quantiles of each real column
    out = pd.DataFrame({col: np.quantile(data[col].values, uniform[:, j], method='inverted_cdf' if col == target else 'linear') for j, col in enumerate(cols)})
    return out

finalize(synth_gaussian_copula(real_train, N_SYNTH), '03 Gaussian Copula').head()

In [ ]:
#Method 04 - KDE by Class ( Smooth Density Estimated from the Real Rows )
def synth_kde_by_class(data, n, seed=SEED):
    rng = np.random.default_rng(seed)
    prior = data[target].value_counts(normalize=True).sort_index()
    counts = rng.multinomial(n, prior.values)
    parts = []
    for cls, k in zip(prior.index, counts):
        kde = stats.gaussian_kde(data.loc[data[target] == cls, num_cols].values.T)
        part = pd.DataFrame(kde.resample(k, seed=rng).T, columns=num_cols)
        part[target] = cls
        parts.append(part)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

finalize(synth_kde_by_class(real_train, N_SYNTH), '04 KDE by Class').head()

In [ ]:
#Method 05 - Smoothed Bootstrap ( Real Rows + Random Noise )
def synth_bootstrap_noise(data, n, noise=0.10, seed=SEED):
    rng = np.random.default_rng(seed)
    out = data.iloc[rng.integers(0, len(data), n)].reset_index(drop=True).copy()
    for col in num_cols:
        out[col] = out[col] + rng.normal(0, noise * data[col].std(), n)    # noise = share of the standard deviation
    return out

finalize(synth_bootstrap_noise(real_train, N_SYNTH), '05 Smoothed Bootstrap').head()

In [ ]:
#Method 06 - SMOTE-like ( New Rows Between a Real Row and One of its Nearest Neighbors )
def smote_interpolate(points, n_new, k=5, seed=SEED):
    rng = np.random.default_rng(seed)
    neighbors = NearestNeighbors(n_neighbors=k + 1).fit(points).kneighbors(points, return_distance=False)[:, 1:]
    base = rng.integers(0, len(points), n_new)
    chosen = neighbors[base, rng.integers(0, k, n_new)]
    gap = rng.uniform(0, 1, (n_new, 1))
    return points[base] + gap * (points[chosen] - points[base])

def synth_smote_like(data, n, k=5, seed=SEED):
    rng = np.random.default_rng(seed)
    scaler = StandardScaler().fit(data[num_cols])
    prior = data[target].value_counts(normalize=True).sort_index()
    counts = rng.multinomial(n, prior.values)
    parts = []
    for cls, m in zip(prior.index, counts):
        points = scaler.transform(data.loc[data[target] == cls, num_cols])
        part = pd.DataFrame(scaler.inverse_transform(smote_interpolate(points, m, k, seed)), columns=num_cols)
        part[target] = cls
        parts.append(part)
    return pd.concat(parts, ignore_index=True).sample(frac=1, random_state=seed).reset_index(drop=True)

finalize(synth_smote_like(real_train, N_SYNTH), '06 SMOTE-like').head()

In [ ]:
#Method 07 - Chained Models ( income and age are Sampled, loan is Predicted from Them, result is Predicted from All )
def synth_chained(data, n, seed=SEED):
    rng = np.random.default_rng(seed)
    out = pd.DataFrame({'income': rng.choice(data['income'].values, n), 'age': rng.choice(data['age'].values, n)})
    # loan = prediction + a real residual ( keeps the noise around the prediction )
    reg = RandomForestRegressor(n_estimators=100, min_samples_leaf=20, oob_score=True, random_state=seed, n_jobs=-1).fit(data[['income', 'age']], data['loan'])
    residuals = data['loan'].values - reg.oob_prediction_
    out['loan'] = reg.predict(out[['income', 'age']]) + rng.choice(residuals, n)
    # result = 1 with the probability predicted by a classifier
    clf = RandomForestClassifier(n_estimators=100, min_samples_leaf=10, random_state=seed, n_jobs=-1).fit(data[num_cols], data[target])
    out[target] = (rng.uniform(size=n) < clf.predict_proba(out[num_cols])[:, 1]).astype(int)
    return out

finalize(synth_chained(real_train, N_SYNTH), '07 Chained Models').head()

In [ ]:
#Method 08 - From Schema ( No Real Data: Distributions and Rules Written by Hand )
schema = {'income': lambda rng, n, built: rng.uniform(20000, 70000, n),
          'age': lambda rng, n, built: rng.uniform(18, 64, n),
          'loan': lambda rng, n, built: rng.gamma(2.0, 1500, n) * (0.5 + built['income'] / 50000),
          'result': lambda rng, n, built: (rng.uniform(size=n) < 1 / (1 + np.exp(-(-2.2 + 0.0004 * (built['loan'] - 4400) - 0.12 * (built['age'] - 40))))).astype(int)}

def synth_from_schema(spec, n, seed=SEED):
    rng = np.random.default_rng(seed)
    built = {}
    for name, generator in spec.items():          # Order matters: a column can use the previous ones
        built[name] = generator(rng, n, built)
    return pd.DataFrame(built)

#The same idea starting from an EMPTY dataframe ( the columns are created first and filled by the schema )
df_empty = pd.DataFrame({col: [pd.NA] * 5 for col in schema})
print(df_empty, '\n')
finalize(synth_from_schema(schema, N_SYNTH), '08 From Schema').head()

In [ ]:
#Bonus - Generate ONLY the Minority Class ( Balance the Classes with Synthetic Rows )
minority = real_train[real_train[target] == 1]
scaler = StandardScaler().fit(minority[num_cols])
n_new = (real_train[target] == 0).sum() - len(minority)                 # Rows needed to have the same size of the majority class
new_rows = pd.DataFrame(scaler.inverse_transform(smote_interpolate(scaler.transform(minority[num_cols]), n_new)), columns=num_cols)
new_rows[target] = 1
balanced = pd.concat([real_train, new_rows], ignore_index=True)

#Metrics ( Class Distribution and Similarity of the New Rows to the Real Minority Class )
print('Positive rate before / after (%):', round(real_train[target].mean() * 100, 2), '/', round(balanced[target].mean() * 100, 2))
print('KS statistic of the new rows vs the real minority class:', {col: round(stats.ks_2samp(minority[col], new_rows[col]).statistic, 3) for col in num_cols})

### **04. Quality Metrics of the Synthetic Data**

A good synthetic dataset needs to be **realistic**, **useful** and **safe**. Each group of metrics answers one question:

| Question | Metric | Ideal | Rule of thumb |
|---|---|---|---|
| Does each column look real? | KS statistic ( mean ) | 0 | < 0.10 |
| | Normalized Wasserstein distance | 0 | < 0.10 |
| | Class distribution gap ( TVD ) | 0 | < 0.02 |
| Are the relationships kept? | Correlation gap ( mean absolute difference ) | 0 | < 0.05 |
| Can a model tell real from synthetic? | Discriminator AUC | 0.5 | < 0.60 |
| Can the synthetic data train a model? | Utility ratio = AUC trained on synthetic / AUC trained on real ( TSTR / TRTR ) | 1 | > 0.90 |
| Is it a copy of the real data? | DCR ratio = median distance synthetic to train / test to train | 1 | close to 1 ( far below 1 = copies ) |
| | Exact copies of real rows | 0 % | 0 % |
| Does it follow the business rules? | Valid rows ( before the correction ) | 100 % | 100 % |

**Quality score** ( heuristic from 0 to 100 to compare methods ): the mean of fidelity ( 1 - KS ), structure ( 1 - correlation gap / 0.2 ), realism ( 1 - 2 x abs( discriminator AUC - 0.5 ) ), utility ( utility ratio, max 1 ) and privacy ( DCR ratio, max 1 ).

In [ ]:
#Define the Metrics of Each Column ( Fidelity )
def fidelity_table(real_df, synth_df):
    rows = {}
    for col in num_cols:
        ks = stats.ks_2samp(real_df[col], synth_df[col])
        rows[col] = {'KS_statistic': ks.statistic, 'KS_pvalue': ks.pvalue,
                     'Wasserstein_norm': stats.wasserstein_distance(real_df[col], synth_df[col]) / real_df[col].std(),
                     'mean_diff_%': (synth_df[col].mean() / real_df[col].mean() - 1) * 100,
                     'std_diff_%': (synth_df[col].std() / real_df[col].std() - 1) * 100}
    return pd.DataFrame(rows).T

def categorical_distance(real_s, synth_s):
    categories = sorted(set(real_s) | set(synth_s))
    p = real_s.value_counts(normalize=True).reindex(categories, fill_value=0).values
    q = synth_s.value_counts(normalize=True).reindex(categories, fill_value=0).values
    return 0.5 * np.abs(p - q).sum(), jensenshannon(p, q, base=2)      # Total variation distance and Jensen-Shannon distance

In [ ]:
#Define the Metrics of the Relationships ( Structure )
def correlation_gap(real_df, synth_df):
    cols = num_cols + [target]
    difference = (real_df[cols].corr() - synth_df[cols].corr()).abs().values
    upper = np.triu_indices(len(cols), k=1)
    return difference[upper].mean(), difference[upper].max()

In [ ]:
#Define the Discriminator ( Can a Model Tell Real from Synthetic? )
def discriminator_auc(real_df, synth_df, seed=SEED):
    cols = num_cols + [target]
    data = pd.concat([real_df[cols], synth_df[cols]], ignore_index=True)
    label = np.r_[np.zeros(len(real_df)), np.ones(len(synth_df))]
    model = RandomForestClassifier(n_estimators=100, min_samples_leaf=5, random_state=seed, n_jobs=-1)
    proba = cross_val_predict(model, data, label, cv=StratifiedKFold(5, shuffle=True, random_state=seed), method='predict_proba')[:, 1]
    return roc_auc_score(label, proba)

In [ ]:
#Define the Utility Metric ( TSTR: Train on Synthetic, Test on Real vs TRTR: Train on Real, Test on Real )
def utility_scores(real_tr, real_te, synth_df, seed=SEED):
    def auc(train_df):
        model = RandomForestClassifier(n_estimators=200, min_samples_leaf=5, random_state=seed, n_jobs=-1).fit(train_df[num_cols], train_df[target])
        return roc_auc_score(real_te[target], model.predict_proba(real_te[num_cols])[:, 1])
    return auc(real_tr), auc(synth_df)

In [ ]:
#Define the Privacy Metrics ( Distance to the Closest Record and Exact Copies )
def privacy_metrics(real_tr, real_te, synth_df):
    scaler = StandardScaler().fit(real_tr[num_cols])
    nearest = NearestNeighbors(n_neighbors=1).fit(scaler.transform(real_tr[num_cols]))
    dcr_synth = np.median(nearest.kneighbors(scaler.transform(synth_df[num_cols]))[0])
    dcr_test = np.median(nearest.kneighbors(scaler.transform(real_te[num_cols]))[0])    # Reference: how far unseen REAL rows are from the train rows
    real_rows = set(map(tuple, real_tr[num_cols].round(6).values))
    copies = np.mean([row in real_rows for row in map(tuple, synth_df[num_cols].round(6).values)]) * 100
    return dcr_synth / dcr_test, copies

In [ ]:
#Define the Function that Evaluates a Synthetic Dataset ( All the Metrics in One Row )
def evaluate_synthetic(name, synth_df, utility=True):
    fid = fidelity_table(real_train, synth_df)
    tvd, js = categorical_distance(real_train[target], synth_df[target])
    corr_mean, corr_max = correlation_gap(real_train, synth_df)
    disc = discriminator_auc(real_train, synth_df)
    dcr_ratio, copies = privacy_metrics(real_train, real_test, synth_df)
    if utility:
        auc_real, auc_synth = utility_scores(real_train, real_test, synth_df)
        utility_ratio = auc_synth / auc_real
    else:
        auc_real = auc_synth = utility_ratio = np.nan
    components = [1 - fid['KS_statistic'].mean(), 1 - min(corr_mean / 0.2, 1), 1 - min(2 * abs(disc - 0.5), 1), min(utility_ratio, 1), min(dcr_ratio, 1)]
    return {'method': name, 'KS_mean': fid['KS_statistic'].mean(), 'Wasserstein_mean': fid['Wasserstein_norm'].mean(), 'class_TVD': tvd,
            'corr_gap_mean': corr_mean, 'corr_gap_max': corr_max, 'discriminator_AUC': disc, 'TSTR_AUC': auc_synth, 'TRTR_AUC': auc_real,
            'utility_ratio': utility_ratio, 'DCR_ratio': dcr_ratio, 'exact_copies_%': copies, 'valid_rows_%': validity.get(name, 100.0),
            'quality_score': np.nanmean(components) * 100}

In [ ]:
#Evaluate the Real Test Set Against the Real Train Set ( Reference: the Best a Generator Can Do, the Noise Floor )
reference = evaluate_synthetic('00 Real Test (reference)', real_test, utility=False)
pd.Series(reference).drop('method').astype(float).round(4)

In [ ]:
#Evaluate All the Methods ( Takes About 30 Seconds )
results = pd.DataFrame([reference] + [evaluate_synthetic(name, data) for name, data in synthetic.items()]).set_index('method')
results.round(3)

In [ ]:
#Rank the Methods by the Quality Score
ranking = results.drop(index='00 Real Test (reference)').sort_values('quality_score', ascending=False)
ranking[['quality_score', 'KS_mean', 'corr_gap_mean', 'discriminator_AUC', 'utility_ratio', 'DCR_ratio', 'exact_copies_%', 'valid_rows_%']].round(3)

In [ ]:
#Chart of the Quality Score
plt.figure(figsize=(9, 4))
sns.barplot(x=ranking['quality_score'], y=ranking.index, color='steelblue')
plt.axvline(results.loc['00 Real Test (reference)', 'quality_score'], color='gray', linestyle='--', label='Real test set (reference)')
plt.xlabel('Quality score ( 0 - 100 )')
plt.ylabel('')
plt.legend(loc='lower right')
plt.tight_layout()
plt.show()

In [ ]:
#Compare the Distributions ( Real vs the Best and the Worst Method )
best, worst = ranking.index[0], ranking.index[-1]
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, col in zip(axes, num_cols):
    sns.kdeplot(real_train[col], ax=ax, label='Real', color='black', linewidth=2)
    sns.kdeplot(synthetic[best][col], ax=ax, label=best, color='tab:green')
    sns.kdeplot(synthetic[worst][col], ax=ax, label=worst, color='tab:red')
    ax.set_title(col)
axes[0].legend()
plt.tight_layout()
plt.show()

In [ ]:
#Compare the Correlation Matrices ( Real vs the Best and the Worst Method )
fig, axes = plt.subplots(1, 3, figsize=(16, 4))
for ax, (title, data) in zip(axes, [('Real', real_train), (best, synthetic[best]), (worst, synthetic[worst])]):
    sns.heatmap(data[num_cols + [target]].corr(), annot=True, fmt='.2f', cmap='coolwarm', vmin=-1, vmax=1, cbar=False, ax=ax)
    ax.set_title(title)
plt.tight_layout()
plt.show()

### **05. Compare the Methods and Choose One**

| Situation | Recommended method |
|---|---|
| Quick test or a baseline | 01 Independent Bootstrap ( attention: it destroys the relationships ) |
| Few numeric columns and roughly normal data | 02 Gaussian by Class |
| Mixed distributions and you need the correlations between numeric columns | 03 Gaussian Copula ( with a binary classifier the correlation is only approximate: check the correlation gap ) |
| Smooth data with few numeric columns | 04 KDE by Class |
| Need data similar to real rows, with some protection | 05 Smoothed Bootstrap ( increase the noise to increase the privacy ) |
| Balance the classes | 06 SMOTE-like on the minority class only |
| Many columns with different types and dependencies | 07 Chained Models |
| There is no real data ( or it cannot be shared ) | 08 From Schema |

**Always check:** privacy ( exact copies and DCR ratio ), the business rules and the quality score. The best method is the one that satisfies the rules of thumb of the metrics, not only the one with the highest score.

In [ ]:
#Check the Rules of Thumb for Each Method ( True = Passed )
checks = pd.DataFrame({'fidelity ( KS < 0.10 )': ranking['KS_mean'] < 0.10,
                       'structure ( corr gap < 0.05 )': ranking['corr_gap_mean'] < 0.05,
                       'realism ( AUC < 0.60 )': ranking['discriminator_AUC'] < 0.60,
                       'utility ( ratio > 0.90 )': ranking['utility_ratio'] > 0.90,
                       'privacy ( no copies )': ranking['exact_copies_%'] == 0,
                       'valid rows ( 100% )': ranking['valid_rows_%'] == 100})
checks['checks_passed'] = checks.sum(axis=1)
checks.sort_values('checks_passed', ascending=False)

In [ ]:
#Choose the Method ( By Default, the One with the Highest Quality Score )
generators = {'01 Independent Bootstrap': synth_independent, '02 Gaussian by Class': synth_gaussian_by_class,
              '03 Gaussian Copula': synth_gaussian_copula, '04 KDE by Class': synth_kde_by_class,
              '05 Smoothed Bootstrap': synth_bootstrap_noise, '06 SMOTE-like': synth_smote_like,
              '07 Chained Models': synth_chained, '08 From Schema': lambda data, n, seed=SEED: synth_from_schema(schema, n, seed)}
chosen_name = ranking.index[0]          # Change here to use another method. Example: '03 Gaussian Copula'
print('Chosen method:', chosen_name)
print(ranking.loc[chosen_name, ['quality_score', 'DCR_ratio', 'exact_copies_%', 'valid_rows_%']].round(3).to_dict())

In [ ]:
#Generate the Final Dataset ( Fit on ALL the Real Data and Create More Rows than the Original )
final_raw = generators[chosen_name](real[num_cols + [target]], 5000, seed=SEED)
final_synthetic = apply_constraints(final_raw, real)
print(final_synthetic.shape)
final_synthetic.head()

In [ ]:
#Final Metrics ( Fidelity of the Final Dataset vs the Real Data )
print(fidelity_table(real, final_synthetic).round(4), '\n')
print('Correlation gap ( mean / max ):', tuple(round(v, 3) for v in correlation_gap(real, final_synthetic)))
print('Positive rate real / synthetic (%):', round(real[target].mean() * 100, 2), '/', round(final_synthetic[target].mean() * 100, 2))
print('Valid rows (%):', round(validity_rate(final_synthetic), 2))

In [ ]:
#Add Categorical Columns to the Synthetic Dataset ( Reuse the Generators of Section 02 )
final_full = add_columns(final_synthetic, column_spec)
final_full['City'] = np.random.choice(list(city_state), size=len(final_full), p=[0.25, 0.10, 0.20, 0.05, 0.15, 0.05, 0.15, 0.05])
final_full['State'] = final_full['City'].map(city_state)

#Metrics ( Shape, Types and Nulls of the Complete Dataset )
print(final_full.shape, '| Nulls:', int(final_full.isnull().sum().sum()))
print(final_full.dtypes.astype(str).to_dict())

In [ ]:
#Save the Synthetic Dataset
final_full.to_csv('synthetic_credit_data.csv', index=False)

#Download the File ( Google Colab )
# from google.colab import files
# files.download('synthetic_credit_data.csv')